# Spectral Joint Flow (SJFPO) on Colab

Train the spectral flow policy on MuJoCo from a clean Colab runtime.
All code lives in `src/` of the cloned repo — this notebook only clones,
installs, and calls `scripts/train.py` (eval runs inside training and lands
in `eval_history.csv`).

## 0. Private-repo access

Create a fine-grained GitHub PAT with **Contents: read** on this repo and
paste it below when prompted (or set the `GITHUB_TOKEN` secret — it is
never written into the notebook). Also set the `WANDB_API_KEY` secret
so wandb login happens automatically (no manual paste).

In [ ]:
# Detect runtime and load secrets from the right store.
import os

def _detect_env():
    try:
        if "google.colab" in str(get_ipython()):
            return "colab"
    except NameError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    return "local"

ENV_NAME = _detect_env()
print(f"Environment: {ENV_NAME}")

def load_secret(key_name):
    try:
        if ENV_NAME == "colab":
            from google.colab import userdata
            return userdata.get(key_name) or None
        if ENV_NAME == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key_name) or None
        return os.getenv(key_name)
    except Exception as e:
        print(f"load_secret({key_name}) failed: {e}")
        return None

In [ ]:
# Clone the private repo (skip if already present), enter it.
# GITHUB_TOKEN comes from the secret store (see cell above); manual paste is the fallback.
import getpass
from pathlib import Path

token = load_secret("GITHUB_TOKEN") or getpass.getpass(
    "GitHub token (PAT with Contents:read on this repo): "
)
repo_url = f"https://{token}@github.com/dzungphieuluuky/spectral-joint-policy.git"
if not Path("spectral-joint-policy").exists():
    !git clone $repo_url
%cd spectral-joint-policy
!git log --oneline -3

In [ ]:
# Install pinned deps with uv (CPU wheels resolve on Colab; GPU torch comes with them).
!pip install -q uv
!uv pip install -q -r requirements-linux.txt --system

In [ ]:
# wandb login from secret store (runs before any training cell).
_wandb_key = load_secret("WANDB_API_KEY")
if _wandb_key:
    !wandb login $_wandb_key
else:
    print("WANDB_API_KEY not found; training will run without wandb sync.")

In [ ]:
# Smoke: 2k steps, single SJFPO run on Hopper (eval + checkpoints included).
!PYTHONPATH=src python scripts/train.py --method sjfpo --env Hopper-v5 --seed 0 --total_steps 2000

In [ ]:
# Full run: 1M steps with the matrix budgets (50k eval / 25k ckpt).
!PYTHONPATH=src python scripts/train.py --method sjfpo --env Hopper-v5 --seed 0 --total_steps 1000000 --eval_steps 50000 --save_steps 25000